In [1]:
# ==============================================================================
# [CELL 1] 2026 FIFA World Cup - Pipeline with Sampling (0:0 무승부 & 이변 발생 패치)
# ==============================================================================
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import shap
import matplotlib.pyplot as plt
from scipy.stats import poisson
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings

# 경고 메시지 숨기기
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---------------------------------------------------------
# 1. Advanced Feature Engineering (Dynamic Elo)
# ---------------------------------------------------------
class AdvancedElo:
    def __init__(self, k=20, base=1500):
        self.k = k
        self.base = base
        self.ratings = {}

    def get_rating(self, team):
        return self.ratings.get(team, self.base)

    def expected_result(self, loc, awc):
        dr = loc - awc
        return 1 / (10 ** (-dr / 400) + 1)

    def update_rating(self, home_team, away_team, home_goals, away_goals, tournament_weight=1.0):
        home_rating = self.get_rating(home_team)
        away_rating = self.get_rating(away_team)

        goal_diff = abs(home_goals - away_goals)
        G = 1.0 if goal_diff <= 1 else (1.5 if goal_diff == 2 else (11 + goal_diff) / 8.0)

        home_expected = self.expected_result(home_rating + 100, away_rating)
        away_expected = self.expected_result(away_rating, home_rating + 100)

        home_actual = 1 if home_goals > away_goals else (0 if home_goals < away_goals else 0.5)
        away_actual = 1 - home_actual

        self.ratings[home_team] = home_rating + self.k * G * tournament_weight * (home_actual - home_expected)
        self.ratings[away_team] = away_rating + self.k * G * tournament_weight * (away_actual - away_expected)

# ---------------------------------------------------------
# 2. Model Training & Visualizing Feature Importance
# ---------------------------------------------------------
def optimize_and_train(X, y, n_trials=30):
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)

    def objective(trial):
        params = {
            'objective': 'poisson', 'metric': 'poisson',
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 63),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 0.9),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 0.9),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 5),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
            'feature_pre_filter': False, 'verbose': -1, 'seed': 42
        }
        gbm = lgb.train(params, train_data, valid_sets=[valid_data], num_boost_round=1000,
                        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)])
        return gbm.best_score['valid_0']['poisson']

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    
    best_params = study.best_params
    best_params.update({'objective': 'poisson', 'feature_pre_filter': False, 'verbose': -1, 'seed': 42})
    
    final_model = lgb.train(best_params, lgb.Dataset(X, label=y), num_boost_round=500)
    return final_model

def visualize_importance(model, X, title):
    print(f"\n📊 [{title}] 피처 중요도 분석 중...")
    plt.figure(figsize=(10, 5))
    lgb.plot_importance(model, importance_type='gain', max_num_features=15, title=f"LightGBM Feature Importance (Gain) - {title}")
    plt.tight_layout()
    plt.show()

    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X)
        plt.figure(figsize=(10, 6))
        plt.title(f"SHAP Summary Plot - {title}")
        shap.summary_plot(shap_values, X, show=False)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"SHAP 그래프 생성 중 에러 발생: {e}")

def train_lgbm_models(df_train, n_trials=30):
    features = ['home_elo', 'away_elo', 'elo_diff', 'home_avg_scored', 'away_avg_conceded',
                'away_avg_scored', 'home_avg_conceded', 'home_form_5', 'away_form_5']
    X = df_train[features]
    
    print("\n[1/2] 홈팀 예측 모델 튜닝 중...")
    lgb_home = optimize_and_train(X, df_train['home_score'], n_trials)
    visualize_importance(lgb_home, X, "Home Goals Model")
    
    print("\n[2/2] 원정팀 예측 모델 튜닝 중...")
    lgb_away = optimize_and_train(X, df_train['away_score'], n_trials)
    visualize_importance(lgb_away, X, "Away Goals Model")
    
    return lgb_home, lgb_away, features

# ---------------------------------------------------------
# 3. Dynamic Simulation Engine (✨업데이트: 확률적 샘플링✨)
# ---------------------------------------------------------
def simulate_match(lambda_home, lambda_away, n_simulations=10000):
    # 0-0 무승부가 다시 발생할 수 있도록 곱해주는 배율을 1.25 -> 1.05로 축소
    lambda_home *= 1.05 
    lambda_away *= 1.05
    
    k_dispersion = 15.0 
    lh_noisy = np.random.gamma(k_dispersion, lambda_home/k_dispersion, size=n_simulations)
    la_noisy = np.random.gamma(k_dispersion, lambda_away/k_dispersion, size=n_simulations)

    hg_sim = poisson.rvs(mu=lh_noisy)
    ag_sim = poisson.rvs(mu=la_noisy)
    
    prob_h = np.sum(hg_sim > ag_sim) / n_simulations
    prob_d = np.sum(hg_sim == ag_sim) / n_simulations
    prob_a = np.sum(hg_sim < ag_sim) / n_simulations
    
    # [핵심 변경 사항] 무조건 1등 스코어만 고르지 않고, 상위 5개 스코어 중 확률에 비례해 하나를 샘플링!
    scores = [f"{h}-{a}" for h, a in zip(hg_sim, ag_sim)]
    unique_scores, counts = np.unique(scores, return_counts=True)
    
    n_top = min(5, len(unique_scores))
    top_indices = np.argsort(counts)[-n_top:]
    top_scores = unique_scores[top_indices]
    top_probs = counts[top_indices] / np.sum(counts[top_indices]) # 확률을 1.0으로 재정규화
    
    predicted_score = np.random.choice(top_scores, p=top_probs)
    pred_h, pred_a = map(int, predicted_score.split('-'))
    
    return prob_h, prob_d, prob_a, pred_h, pred_a

def get_live_features(t1, t2, team_stats, elo_system):
    def get_stat(t, key): return np.mean(team_stats[t][key][-10:]) if team_stats.get(t, {}).get(key) else 1.0
    def get_form(t): return sum(team_stats[t]['pts'][-5:]) if team_stats.get(t, {}).get('pts') else 0

    return pd.DataFrame([{
        'home_elo': elo_system.get_rating(t1), 'away_elo': elo_system.get_rating(t2),
        'elo_diff': elo_system.get_rating(t1) - elo_system.get_rating(t2),
        'home_avg_scored': get_stat(t1, 'scored'), 'home_avg_conceded': get_stat(t1, 'conceded'),
        'away_avg_scored': get_stat(t2, 'scored'), 'away_avg_conceded': get_stat(t2, 'conceded'),
        'home_form_5': get_form(t1), 'away_form_5': get_form(t2)
    }])

# ---------------------------------------------------------
# 4. Execution Pipeline
# ---------------------------------------------------------
if __name__ == "__main__":
    print("🚀 월드컵 데이터 로드 및 피처 생성 시작...\n")
    
    FILE_PATH = "/kaggle/input/datasets/dybalar/feature/historical_results.csv"
    OUTPUT_PATH = "submission_consensus_fixed.csv"
    
    name_corrections = {
        'Cura?ao': 'Curaçao', 'Curacao': 'Curaçao', 'DR Congo': 'Congo DR', 
        'Cape Verde': 'Cape Verde Islands', 'Czech Republic': 'Czechia',
        'Bosnia and Herzegovina': 'Bosnia-Herzegovina', 'Bosnia': 'Bosnia-Herzegovina'
    }

    df = pd.read_csv(FILE_PATH)
    df['home_team'] = df['home_team'].replace(name_corrections)
    df['away_team'] = df['away_team'].replace(name_corrections)
    df['home_score'] = pd.to_numeric(df['home_score'], errors='coerce')
    df['away_score'] = pd.to_numeric(df['away_score'], errors='coerce')
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)

    elo_system = AdvancedElo()
    home_elos, away_elos, h_avg_s, h_avg_c, a_avg_s, a_avg_c, h_form, a_form = [], [], [], [], [], [], [], []
    team_stats = {}

    for idx, row in df.iterrows():
        ht, at = row['home_team'], row['away_team']
        home_elos.append(elo_system.get_rating(ht)); away_elos.append(elo_system.get_rating(at))
        for t in (ht, at):
            if t not in team_stats: team_stats[t] = {'scored': [], 'conceded': [], 'pts': []}
                
        if len(team_stats[ht]['scored']) > 0:
            h_avg_s.append(np.mean(team_stats[ht]['scored'][-10:])); h_avg_c.append(np.mean(team_stats[ht]['conceded'][-10:]))
            h_form.append(sum(team_stats[ht]['pts'][-5:]))
        else:
            h_avg_s.append(1.0); h_avg_c.append(1.0); h_form.append(0)
            
        if len(team_stats[at]['scored']) > 0:
            a_avg_s.append(np.mean(team_stats[at]['scored'][-10:])); a_avg_c.append(np.mean(team_stats[at]['conceded'][-10:]))
            a_form.append(sum(team_stats[at]['pts'][-5:]))
        else:
            a_avg_s.append(1.0); a_avg_c.append(1.0); a_form.append(0)

        if pd.notna(row['home_score']) and pd.notna(row['away_score']):
            hs, ast = float(row['home_score']), float(row['away_score'])
            elo_system.update_rating(ht, at, hs, ast)
            team_stats[ht]['scored'].append(hs); team_stats[ht]['conceded'].append(ast)
            team_stats[at]['scored'].append(ast); team_stats[at]['conceded'].append(hs)
            team_stats[ht]['pts'].append(3 if hs > ast else (0 if hs < ast else 1))
            team_stats[at]['pts'].append(0 if hs > ast else (3 if hs < ast else 1))

    df['home_elo'] = home_elos; df['away_elo'] = away_elos; df['elo_diff'] = df['home_elo'] - df['away_elo']
    df['home_avg_scored'] = h_avg_s; df['home_avg_conceded'] = h_avg_c
    df['away_avg_scored'] = a_avg_s; df['away_avg_conceded'] = a_avg_c
    df['home_form_5'] = h_form; df['away_form_5'] = a_form

    mask_2026_wc = (df['date'].dt.year == 2026) & (df['tournament'] == 'FIFA World Cup')
    df_train = df[~mask_2026_wc].dropna(subset=['home_score', 'away_score']).copy()
    df_group = df[mask_2026_wc].copy() 

    OPTUNA_TRIALS = 30
    lgb_home, lgb_away, feature_cols = train_lgbm_models(df_train, n_trials=OPTUNA_TRIALS)

    lgb_home.save_model('lgb_home_model.txt')
    lgb_away.save_model('lgb_away_model.txt')
    print("\n💾 학습된 LightGBM 모델이 저장되었습니다! (lgb_home_model.txt, lgb_away_model.txt)")

    print("\n🏃‍♂️ [STAGE 1] 조별리그 72경기 시뮬레이션 중...")
    results = []
    df_group['lambda_home'] = lgb_home.predict(df_group[feature_cols])
    df_group['lambda_away'] = lgb_away.predict(df_group[feature_cols])
    
    for idx, row in df_group.iterrows():
        p_home, p_draw, p_away, pred_h, pred_a = simulate_match(row['lambda_home'], row['lambda_away'])
        t1_prob = p_home + p_draw * (p_home / (p_home + p_away + 1e-9))
        t2_prob = p_away + p_draw * (p_away / (p_home + p_away + 1e-9))
        results.append({
            'team1': row['home_team'], 'team2': row['away_team'], 'team1_score': int(pred_h), 'team2_score': int(pred_a),
            'team1_prob': round(t1_prob, 4), 'team2_prob': round(t2_prob, 4), 'type': 'group'
        })
    
    print("🏆 [STAGE 2] 순위 집계 및 32강~결승전 토너먼트 자동 생성 중...")
    WC_GROUPS = {
        'A': ['Mexico', 'South Africa', 'South Korea', 'Czechia'], 'B': ['Canada', 'Bosnia-Herzegovina', 'United States', 'Paraguay'],
        'C': ['Qatar', 'Switzerland', 'Brazil', 'Morocco'], 'D': ['Haiti', 'Scotland', 'Australia', 'Turkey'],
        'E': ['Germany', 'Curaçao', 'Netherlands', 'Japan'], 'F': ['Ivory Coast', 'Ecuador', 'Sweden', 'Tunisia'],
        'G': ['Spain', 'Cape Verde Islands', 'Belgium', 'Egypt'], 'H': ['Saudi Arabia', 'Uruguay', 'Iran', 'New Zealand'],
        'I': ['France', 'Senegal', 'Iraq', 'Norway'], 'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
        'K': ['Portugal', 'Congo DR', 'England', 'Croatia'], 'L': ['Ghana', 'Panama', 'Uzbekistan', 'Colombia']
    }
    
    standings = {team: {'pts': 0, 'gd': 0, 'gf': 0, 'group': g} for g, teams in WC_GROUPS.items() for team in teams}
    for r in results:
        t1, t2, s1, s2 = r['team1'], r['team2'], r['team1_score'], r['team2_score']
        if t1 in standings and t2 in standings:
            standings[t1]['gf'] += s1; standings[t2]['gf'] += s2
            standings[t1]['gd'] += (s1 - s2); standings[t2]['gd'] += (s2 - s1)
            if s1 > s2: standings[t1]['pts'] += 3
            elif s1 < s2: standings[t2]['pts'] += 3
            else: standings[t1]['pts'] += 1; standings[t2]['pts'] += 1

    group_rankings = {g: sorted([t for t, s in standings.items() if s['group'] == g], key=lambda x: (standings[x]['pts'], standings[x]['gd'], standings[x]['gf']), reverse=True) for g in WC_GROUPS}
    
    thirds = sorted([group_rankings[g][2] for g in WC_GROUPS], key=lambda x: (standings[x]['pts'], standings[x]['gd'], standings[x]['gf']), reverse=True)[:8]
    r32_teams = [group_rankings[g][0] for g in WC_GROUPS] + [group_rankings[g][1] for g in WC_GROUPS] + thirds
    
    current_round_teams = r32_teams[:32]
    
    while len(current_round_teams) > 1:
        next_round_teams = []
        for i in range(0, len(current_round_teams), 2):
            t1, t2 = current_round_teams[i], current_round_teams[i+1]
            match_features = get_live_features(t1, t2, team_stats, elo_system)
            
            l_home = lgb_home.predict(match_features)[0]
            l_away = lgb_away.predict(match_features)[0]
            
            p_home, p_draw, p_away, pred_h, pred_a = simulate_match(l_home, l_away)
            t1_prob = p_home + p_draw * (p_home / (p_home + p_away + 1e-9))
            t2_prob = p_away + p_draw * (p_away / (p_home + p_away + 1e-9))
            
            # 토너먼트 룰: 무승부일 경우 강제 타이브레이커 (승률에 기반해 1점 추가)
            if pred_h == pred_a:
                if t1_prob > t2_prob: pred_h += 1
                else: pred_a += 1
                
            results.append({
                'team1': t1, 'team2': t2, 'team1_score': int(pred_h), 'team2_score': int(pred_a),
                'team1_prob': round(t1_prob, 4), 'team2_prob': round(t2_prob, 4), 'type': 'knockout'
            })
            next_round_teams.append(t1 if pred_h > pred_a else t2)
            
        current_round_teams = next_round_teams

    df_final = pd.DataFrame(results, columns=['team1', 'team2', 'team1_score', 'team2_score', 'team1_prob', 'team2_prob', 'type'])
    df_final.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
    print(f"\n✅ 대진표 완성! 총 103경기 예측 파일 저장 완료: {OUTPUT_PATH}")

ModuleNotFoundError: No module named 'lightgbm'

In [ ]:
# ==============================================================================
# [CELL 2] 제출 파일 분석기 (Score Analyzer)
# 모델이 지나치게 정배(안전한 예측)만 하진 않았는지 평가합니다.
# ==============================================================================
import pandas as pd

def analyze_submission(file_path):
    print("📊 제출 파일(Submission) 분석 시작...\n" + "="*50)
    
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {file_path}")
        return

    # 1. 104경기 정상 생성 확인
    total_matches = len(df)
    group_matches = len(df[df['type'] == 'group'])
    knockout_matches = len(df[df['type'] == 'knockout'])
    print(f"✅ 총 경기 수: {total_matches}경기 (조별리그 {group_matches} / 토너먼트 {knockout_matches})")
    
    if knockout_matches == 32:
        print("  -> 결승전까지의 32개 토너먼트 대진표가 완벽하게 생성되었습니다.")
    
    # 2. 득점 분포 교정 확인
    df['total_goals'] = df['team1_score'] + df['team2_score']
    avg_goals = df['total_goals'].mean()
    zero_zero_draws = len(df[(df['team1_score'] == 0) & (df['team2_score'] == 0) & (df['type'] == 'group')])
    high_scoring = len(df[df['total_goals'] >= 5])
    
    print("\n[득점 분포 진단]")
    print(f" - 평균 양 팀 합계 득점: {avg_goals:.2f}골 (정상범위: 2.3 ~ 2.8골)")
    print(f" - 조별리그 0:0 무승부 예측 경기 수: {zero_zero_draws}경기")
    print(f" - 5골 이상 다득점 경기: {high_scoring}경기")

    # 3. 이변(Upset)과 극단성 교정 확인
    extreme_probs = df[(df['team1_prob'] >= 0.90) | (df['team1_prob'] <= 0.10)]
    upset_predictions = df[((df['team1_prob'] < 0.40) & (df['team1_score'] > df['team2_score'])) | 
                           ((df['team2_prob'] < 0.40) & (df['team2_score'] > df['team1_score']))]
    
    print("\n[확률 캘리브레이션 진단]")
    print(f" - 90% 이상 극단적 확률 부여 경기: {len(extreme_probs)}경기 (이 수치가 너무 높으면 과적합 위험)")
    print(f" - 역배(Upset)를 만들어낸 경기 수: {len(upset_predictions)}경기 (이변 창출이 있어야 좋은 점수를 받음)")

    # 4. 결승전 진출국 추적
    if knockout_matches > 0:
        final_match = df[df['type'] == 'knockout'].iloc[-1]
        print("\n[🏆 모델이 예측한 결승전 매치업]")
        print(f" - {final_match['team1']} ({final_match['team1_prob']*100:.1f}%) VS {final_match['team2']} ({final_match['team2_prob']*100:.1f}%)")
        print(f" - 결승전 예측 스코어: {final_match['team1']} {final_match['team1_score']} : {final_match['team2_score']} {final_match['team2']}")

    print("\n" + "="*50)

# 실행
if __name__ == "__main__":
    submission_file = "submission_consensus_fixed.csv"
    analyze_submission(submission_file)